# 🧠 CanScan - Brain Tumor Classification using GoogLeNet v3 (InceptionV3)

This notebook implements the exact **CanScan** classification architecture for brain tumor MRI scans.

### 🎯 Target Classes (4 classes):
- `glioma`
- `meningioma`
- `notumor`
- `pituitary`

### 🏗️ Model Architecture:
- **Backbone**: `InceptionV3` (`input_shape=(224, 224, 3)`, pre-trained on ImageNet)
- **Classification Head**:
  - `GlobalAveragePooling2D`
  - `BatchNormalization`
  - `Dense(256, activation='relu')`
  - `Dropout(0.4)`
  - `Dense(4, activation='softmax')`

### ⚡ Training Strategy:
1. **Phase 1 (Transfer Learning)**: Backbone frozen, Adam ($lr=10^{-4}$), 10 epochs.
2. **Phase 2 (Fine-Tuning)**: Unfreeze last 50 layers of InceptionV3 (keeping BatchNormalization frozen), Adam ($lr=10^{-5}$), 15 epochs.
3. **Regularization & Callbacks**: `EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`.
4. **Evaluation**: Test accuracy & loss, Confusion Matrix, and Classification Report.

In [ ]:
# =============================================================================
# 1. Environment & Hardware Verification
# =============================================================================
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers, models, applications, callbacks, optimizers
from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU Detected: {len(gpus)} device(s) -> {gpus[0].name}")
else:
    print("\n⚠️ No GPU detected. In Kaggle, enable GPU via Settings -> Accelerator -> GPU T4 / P100.")

## 2. Locate the Processed Dataset on Kaggle
This helper scans `/kaggle/input/` to automatically find the folder containing the `train/`, `val/`, and `test/` splits.

In [ ]:
# =============================================================================
# 2. Automatic Dataset Discovery
# =============================================================================
def find_dataset_root(search_root="/kaggle/input"):
    """Recursively find the directory that contains 'train', 'val', and 'test' subdirectories."""
    if not os.path.exists(search_root):
        return None
    for root, dirs, _ in os.walk(search_root):
        if {"train", "val", "test"}.issubset(set(dirs)):
            return root
    return None

DATASET_ROOT = find_dataset_root()

# Fallback for local testing or custom path
if DATASET_ROOT is None:
    local_path = "data/classification"
    if os.path.exists(local_path) and {"train", "val", "test"}.issubset(set(os.listdir(local_path))):
        DATASET_ROOT = local_path
    else:
        # Replace this with your specific dataset path if different
        DATASET_ROOT = "/kaggle/input/canscan-classification"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

print(f"📁 Using Dataset Root: {DATASET_ROOT}")
print(f"  • Train directory: {TRAIN_DIR}")
print(f"  • Val directory:   {VAL_DIR}")
print(f"  • Test directory:  {TEST_DIR}")

## 3. Verify Classes and Split Distribution

In [ ]:
# =============================================================================
# 3. Distribution Inspection
# =============================================================================
splits = ["train", "val", "test"]
classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith(".")])

records = []
print("Sample counts per split:")
for s in splits:
    s_dir = os.path.join(DATASET_ROOT, s)
    counts = {}
    for c in classes:
        c_path = os.path.join(s_dir, c)
        n = len(os.listdir(c_path)) if os.path.exists(c_path) else 0
        counts[c] = n
        records.append({"Split": s.capitalize(), "Class": c, "Count": n})
    print(f"  • {s.capitalize():5s}: Total = {sum(counts.values()):5d} -> {counts}")

# Barplot visualization
df_counts = pd.DataFrame(records)
plt.figure(figsize=(10, 4.5))
sns.barplot(data=df_counts, x="Class", y="Count", hue="Split", palette="Set2")
plt.title("Distribution of Images per Split and Class", fontsize=13, fontweight="bold")
plt.ylabel("Number of Images")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Build TensorFlow Datasets (`tf.data`)
We load batches of $224 \times 224$ images, rescale pixel values to $[0, 1]$, and use `tf.data.AUTOTUNE` for fast prefetching.

In [ ]:
# =============================================================================
# 4. Data Loading Pipeline
# =============================================================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(classes)

rescaling = layers.Rescaling(1.0 / 255)

train_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_raw = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_raw = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_raw.class_names
print(f"\nDetected Class Names: {class_names}")

# Apply Rescaling & Prefetching
train_ds = train_raw.map(lambda x, y: (rescaling(x), y)).prefetch(tf.data.AUTOTUNE)
val_ds   = val_raw.map(lambda x, y: (rescaling(x), y)).prefetch(tf.data.AUTOTUNE)
test_ds  = test_raw.map(lambda x, y: (rescaling(x), y)).prefetch(tf.data.AUTOTUNE)

print("✅ tf.data datasets compiled and ready!")

## 5. Preview Input Batch

In [ ]:
# Visualize sample training images
plt.figure(figsize=(12, 6))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[labels[i].numpy()], fontsize=11, fontweight="bold")
        plt.axis("off")
plt.suptitle("Sample Normalized Input Images (224x224)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Build the GoogLeNet v3 Architecture
We construct the InceptionV3 backbone with the custom classification head.

In [ ]:
# =============================================================================
# 6. Model Definition (CanScan GoogLeNetV3)
# =============================================================================
def build_googlenetv3(num_classes=4, input_shape=(224, 224, 3), lr=1e-4):
    """Constructs and compiles GoogLeNet v3 (InceptionV3) for classification."""
    base_model = applications.InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )
    # Freeze base model initially
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape, name="input_image")
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.4, name="dropout_0.4")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = models.Model(inputs, outputs, name="GoogLeNetV3")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base_model

model, base_model = build_googlenetv3(num_classes=NUM_CLASSES)
model.summary()

## 7. Setup Training Callbacks
- **`EarlyStopping`**: Stops training if `val_loss` does not improve for 5 epochs.
- **`ModelCheckpoint`**: Saves the weights of the best epoch to `/kaggle/working/best_googlenetv3.keras`.
- **`ReduceLROnPlateau`**: Reduces learning rate by $5\times$ when validation loss plateaus.

In [ ]:
# =============================================================================
# 7. Callbacks Setup
# =============================================================================
CHECKPOINT_PATH = "/kaggle/working/best_googlenetv3.keras" if os.path.exists("/kaggle") else "models/best_googlenetv3.keras"
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

callbacks_list = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
]
print(f"Best model will be saved at: {CHECKPOINT_PATH}")

## 8. Phase 1: Transfer Learning (Feature Extraction)
Train the newly initialized top classification layers for 10 epochs while keeping the base feature extractor frozen.

In [ ]:
# =============================================================================
# 8. Phase 1 Training
# =============================================================================
EPOCHS_PHASE1 = 10

print(f"🚀 Starting Phase 1 Training (1 to {EPOCHS_PHASE1} epochs)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_list
)

## 9. Phase 2: Fine-Tuning
We unfreeze the top 50 layers of the `InceptionV3` backbone (leaving `BatchNormalization` frozen to preserve learned mean/variance stats) and recompile with a lower learning rate ($10^{-5}$).

In [ ]:
# =============================================================================
# 9. Phase 2 Setup & Training
# =============================================================================
def setup_fine_tuning(model, base_model, unfreeze_from=-50, lr=1e-5):
    """Unfreezes the top layers of the backbone and recompiles."""
    base_model.trainable = True

    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False

    for layer in base_model.layers[unfreeze_from:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

setup_fine_tuning(model, base_model, unfreeze_from=-50, lr=1e-5)

trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"✅ Model recompiled for Fine-Tuning. Trainable weights: {trainable_params:,}")

TOTAL_EPOCHS = 25
print(f"🚀 Starting Phase 2 Training (Epochs {EPOCHS_PHASE1} to {TOTAL_EPOCHS})...")

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=EPOCHS_PHASE1,
    epochs=TOTAL_EPOCHS,
    callbacks=callbacks_list
)

## 10. Training Convergence Curves
Plotting training vs validation accuracy and loss across both phases.

In [ ]:
# Combine phase 1 and phase 2 histories
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Accuracy', color='#1f77b4', lw=2)
plt.plot(epochs_range, val_acc, label='Val Accuracy', color='#ff7f0e', lw=2)
plt.axvline(x=EPOCHS_PHASE1, color='green', linestyle='--', label='Fine-Tuning Start')
plt.title('Accuracy Across Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)

# Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss', color='#1f77b4', lw=2)
plt.plot(epochs_range, val_loss, label='Val Loss', color='#ff7f0e', lw=2)
plt.axvline(x=EPOCHS_PHASE1, color='green', linestyle='--', label='Fine-Tuning Start')
plt.title('Loss Across Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 11. Final Evaluation on Held-Out Test Set
Load the best checkpoint weights and evaluate performance.

In [ ]:
# Load best checkpoint model
print(f"Loading best model weights from: {CHECKPOINT_PATH}")
best_model = keras.models.load_model(CHECKPOINT_PATH)

# Evaluate on test split
test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f"\n🎯 Test Accuracy: {test_acc * 100:.2f}%")
print(f"🎯 Test Loss:     {test_loss:.4f}")

# Generate predictions for metrics
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.argmax(np.array(y_pred_probs), axis=1)

# Classification Report
print("\n" + "="*60)
print("CLASSIFICATION REPORT (TEST SET):")
print("="*60)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Sample Counts)', fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.subplot(1, 2, 2)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix (%)', fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.tight_layout()
plt.show()

## 12. Visual Predictions on Test Images

In [ ]:
plt.figure(figsize=(14, 7))
for images, labels in test_ds.take(1):
    preds = best_model.predict(images, verbose=0)
    for i in range(min(8, len(images))):
        ax = plt.subplot(2, 4, i + 1)
        img = images[i].numpy()
        true_label = class_names[labels[i].numpy()]
        pred_label = class_names[np.argmax(preds[i])]
        confidence = np.max(preds[i]) * 100
        
        is_correct = (true_label == pred_label)
        color = "green" if is_correct else "red"
        
        plt.imshow(img)
        plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)", color=color, fontsize=10, fontweight="bold")
        plt.axis("off")

plt.suptitle("Sample Predictions on Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 13. Export Final Model
Save the final model in `.keras` format to `/kaggle/working/` so you can download it directly from Kaggle's **Output** panel.

In [ ]:
# Save model to working directory for easy download from Kaggle output
output_dir = "/kaggle/working" if os.path.exists("/kaggle") else "models"
final_model_path = os.path.join(output_dir, "classifier.keras")
best_model.save(final_model_path)

print(f"🎉 Model saved to: {final_model_path}")
print(f"📦 File size: {os.path.getsize(final_model_path) / (1024 * 1024):.2f} MB")